# dev-fingerprint — Exploration Notebook

**Behavioral fingerprinting of open-source contributors**

This notebook walks through the full analysis pipeline at two levels:

**Part A — Synthetic demo** (no GitHub token needed)
1. Style signal extraction from commit patches (Level C)
2. Quarterly aggregation into BehaviorWindows
3. Change-point detection correlated with LLM milestones
4. Multi-developer process comparison (synthetic archetypes)
5. Radar chart — style profile

**Part B — Real data** (reads from `reports/real/` — no token needed)
6. Load and visualize real Level-A process signals
7. Statistical drift results — all 9 developers
8. Signal-level detail for top drifters

> **Data:** 4,926 commits · 9 developers · 2018–2025 · 120 commits/year uniform sampling  
> **Test:** Mann-Whitney U per Level-A signal, Fisher combined p-value  
> **Minimum:** 10 quarterly windows required for statistical verdict

In [ ]:
import sys, json
from pathlib import Path

sys.path.insert(0, "..")

from datetime import datetime, timezone
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from devfp.models import (
    BehaviorWindow, Commit, CommitFile, Language, LLM_MILESTONES,
)
from devfp.analyzer.style import extract_metrics
from devfp.analyzer.llm_signals import aggregate_quarterly
from devfp.analyzer.temporal import detect_change_points, _nearest_milestone

REPORTS_DIR = Path("../reports/real")

print("✓ imports OK")
print(f"  LLM milestones: {len(LLM_MILESTONES)} events")
print(f"  Real profiles available: {len(list(REPORTS_DIR.glob('*.json')) if REPORTS_DIR.exists() else [])} JSON files")

## Part A — Synthetic Demo

### 1. Style Signal Extraction (Level C)

We feed raw commit patches through `extract_metrics()` and compare a *human-style* snippet
vs an *LLM-style* snippet. These are **Level-C signals** — informative but not primary evidence
(see `CRITIQUE.md` for why).

In [ ]:
HUMAN_PATCH = """\
+def calc(x, y):
+    return x + y
+
+def proc(lst):
+    res = []
+    for i in lst:
+        if i > 0:
+            res.append(i * 2)
+    return res
+
+# TODO: fix edge case
+def get_user(id):
+    try:
+        return db.find(id)
+    except:
+        return None
"""

LLM_PATCH = """\
+def calculate_sum(first_value: int, second_value: int) -> int:
+    \"\"\"
+    Calculate the sum of two integers.
+
+    Args:
+        first_value: The first integer to add.
+        second_value: The second integer to add.
+
+    Returns:
+        The sum of first_value and second_value.
+    \"\"\"
+    return first_value + second_value
+
+def process_positive_items(items: list[int]) -> list[int]:
+    \"\"\"Filter and double all positive items in the list.\"\"\"
+    result = []
+    for item in items:
+        if item > 0:
+            result.append(item * 2)
+    return result
+
+def get_user_by_id(user_id: int) -> Optional[User]:
+    \"\"\"Retrieve a user from the database by their ID.\"\"\"
+    try:
+        return database.find_by_id(user_id)
+    except DatabaseError as error:
+        logger.error(f\"Failed to fetch user {user_id}: {error}\")
+        return None
"""

def _make_commit(patch, sha="x", date=None):
    return Commit(
        sha=sha, author="dev",
        date=date or datetime(2023, 6, 1, tzinfo=timezone.utc),
        message="feat: update",
        files=[CommitFile(filename="f.py", language=Language.PYTHON,
                          additions=20, deletions=0, patch=patch)],
    )

m_human = extract_metrics(_make_commit(HUMAN_PATCH, sha="h"))
m_llm   = extract_metrics(_make_commit(LLM_PATCH,   sha="l"))

signals = ["comment_density", "docstring_coverage", "avg_identifier_length",
           "error_handling_density"]
labels  = ["Comment density", "Docstring coverage", "Identifier length", "Error handling"]

fig = go.Figure()
for metrics, name, color in [
    (m_human, "Human-style commit (organic)",    "#4a90d9"),
    (m_llm,   "LLM-assisted style (synthetic)",  "#e74c3c"),
]:
    values = [getattr(metrics, s, 0) for s in signals]
    fig.add_trace(go.Bar(name=name, x=labels, y=values,
                         marker_color=color, opacity=0.85))

fig.update_layout(
    title="Level-C style signals: human vs LLM-assisted commit patch",
    barmode="group", yaxis_title="Score",
    legend=dict(orientation="h", y=1.12),
    template="plotly_white", height=380,
    annotations=[dict(
        x=0.5, y=-0.18, xref="paper", yref="paper",
        text="⚠ Level-C signals are informative but not primary evidence — see CRITIQUE.md",
        showarrow=False, font=dict(size=10, color="#888"),
    )],
)
fig.show()

### 2. Synthetic BehaviorWindows — Three Process Archetypes

We simulate 3 archetypes to demonstrate the quarterly aggregation and change-point detection pipeline:

- **`stable`** — no process drift, steady commit rhythm (analogous to Rich Harris in real data)
- **`withdrawing`** — gradual activity decline post-2022 (analogous to Dan Abramov — commits/week collapse)
- **`restructuring`** — sharp structural shift mid-period, then stable (analogous to Sindre Sorhus)

These are **deliberately synthetic** and exaggerated for illustration. Real values are in Part B.

In [ ]:
rng = np.random.default_rng(42)

quarters = [(y, q) for y in range(2019, 2026) for q in range(1, 5)]
quarters = [qt for qt in quarters if not (qt[0] == 2025 and qt[1] > 2)]
N = len(quarters)

def make_window(author, year, q, cpw, files_pc, xmod, refactor, inter_h, style):
    sm = (q - 1) * 3 + 1
    return BehaviorWindow(
        author=author,
        period_start=datetime(year, sm, 1, tzinfo=timezone.utc),
        period_end=datetime(year, sm + 2, 28, tzinfo=timezone.utc),
        n_commits=max(4, int(cpw * 13)),
        commits_per_week=max(0.05, float(cpw)),
        median_files_per_commit=float(files_pc),
        cross_module_ratio=float(np.clip(xmod, 0, 1)),
        refactor_ratio=float(np.clip(refactor, 0, 1)),
        median_inter_commit_hours=float(inter_h),
        median_net_lines=20.0,
        large_commit_ratio=0.05,
        test_touch_ratio=0.2,
        merge_ratio=0.0,
        style_score=float(style),
    )

copilot_idx = next(i for i, (y, q) in enumerate(quarters) if y == 2022 and q == 3)

# Stable: constant high activity, no change
stable_windows = [
    make_window("stable", y, q,
        cpw=rng.normal(4.5, 0.4),
        files_pc=rng.normal(3, 0.5),
        xmod=rng.normal(0.18, 0.03),
        refactor=rng.normal(0.10, 0.02),
        inter_h=rng.normal(30, 5),
        style=rng.normal(8, 1.5),
    )
    for i, (y, q) in enumerate(quarters)
]

# Withdrawing: sharp activity collapse starting at Copilot GA
def withdrawing_cpw(i):
    if i < copilot_idx: return rng.normal(4.5, 0.4)
    return max(0.05, rng.normal(4.5, 0.4) * np.exp(-0.3 * (i - copilot_idx)))

withdrawing_windows = [
    make_window("withdrawing", y, q,
        cpw=withdrawing_cpw(i),
        files_pc=rng.normal(3, 0.5),
        xmod=max(0, rng.normal(0.18, 0.03) - 0.02 * max(0, i - copilot_idx)),
        refactor=max(0, rng.normal(0.10, 0.02) - 0.01 * max(0, i - copilot_idx)),
        inter_h=rng.normal(30, 5) * (1 + 8 * max(0, i - copilot_idx) / N),
        style=rng.normal(8, 1.5),
    )
    for i, (y, q) in enumerate(quarters)
]

# Restructuring: shift in commit structure (larger, more cross-module) from 2023
shift_idx = next(i for i, (y, q) in enumerate(quarters) if y == 2023 and q == 1)
restructuring_windows = [
    make_window("restructuring", y, q,
        cpw=rng.normal(2.5, 0.5),
        files_pc=rng.normal(1.2 if i < shift_idx else 3.5, 0.3),
        xmod=rng.normal(0.05 if i < shift_idx else 0.35, 0.02),
        refactor=rng.normal(0.15, 0.03),
        inter_h=rng.normal(120, 20),
        style=rng.normal(6, 1.2),
    )
    for i, (y, q) in enumerate(quarters)
]

timelines = {
    "stable":         stable_windows,
    "withdrawing":    withdrawing_windows,
    "restructuring":  restructuring_windows,
}

print(f"Generated {N} quarters × 3 archetypes = {N*3} BehaviorWindows")
print(f"Period: {quarters[0]} → {quarters[-1]}")

### 3. Process Signal Timelines + Change-Point Detection

We plot `commits_per_week` (Level-A signal) for each archetype and overlay
detected change points (PELT + CUSUM) and LLM release milestones.

In [ ]:
COLORS = {"stable": "#27ae60", "withdrawing": "#e74c3c", "restructuring": "#f39c12"}
MILESTONE_COLORS = {
    "GitHub Copilot GA":          "#2980b9",
    "ChatGPT Launch":             "#8e44ad",
    "GPT-4 Release":              "#c0392b",
    "GitHub Copilot Chat GA":     "#16a085",
    "Claude 3 Opus":              "#d35400",
}

fig = go.Figure()

# LLM milestone vertical lines
for label, dt in LLM_MILESTONES.items():
    if label not in MILESTONE_COLORS:
        continue
    fig.add_vline(
        x=dt.timestamp() * 1000,
        line_dash="dot", line_color=MILESTONE_COLORS[label], line_width=1.5,
        annotation_text=label.replace("GitHub ", "").replace(" Launch", ""),
        annotation_position="top",
        annotation_font_size=9,
    )

for author, windows in timelines.items():
    xs  = [w.period_start for w in windows]
    ys  = [w.commits_per_week for w in windows]
    color = COLORS[author]

    fig.add_trace(go.Scatter(
        x=xs, y=ys, mode="lines+markers", name=author,
        line=dict(color=color, width=2),
        marker=dict(size=5),
    ))

    # Run change-point detection on commits_per_week signal
    cps = detect_change_points(author, windows, signals=["commits_per_week"])
    for cp in cps:
        fig.add_trace(go.Scatter(
            x=[cp.date], y=[cp.value_after + 0.3],
            mode="markers",
            marker=dict(symbol="triangle-down", size=14, color=color,
                        line=dict(width=1, color="white")),
            name=f"{author} — change point", showlegend=False,
            hovertext=f"{author}: Δ {cp.magnitude:+.2f} cpw\n{cp.nearest_known_event or '—'}",
        ))

fig.update_layout(
    title="commits/week — 3 synthetic archetypes with PELT+CUSUM change-point detection",
    xaxis_title="Quarter",
    yaxis_title="commits / week",
    legend=dict(orientation="h", y=-0.18),
    template="plotly_white",
    height=480,
)
fig.show()

print("\nChange points detected (commits_per_week):")
for author, windows in timelines.items():
    cps = detect_change_points(author, windows, signals=["commits_per_week"])
    print(f"  {author:15s}: {len(cps)} CPs")

### 4. Real Level-A Process Summary — All 9 Developers

This cell loads the **real profiles** from `reports/real/summary.json` and displays the
Level-A Fisher combined p-values, number of quarterly windows, and Level-A change point counts.

**Signal hierarchy reminder:**
- Level A = process signals from commit metadata (primary evidence)
- Level C = style signals from AST analysis (not primary evidence)

In [ ]:
summary_path = REPORTS_DIR / "summary.json"
if not summary_path.exists():
    print("⚠ reports/real/summary.json not found. Run: python run_analysis.py")
else:
    summary = json.loads(summary_path.read_text())

    # Sort: tested first (by p-value), then untested
    tested   = sorted([s for s in summary if s["drift_combined_p"] is not None],
                      key=lambda s: s["drift_combined_p"])
    untested = [s for s in summary if s["drift_combined_p"] is None]
    ordered  = tested + untested

    names     = [s["display_name"] for s in ordered]
    p_vals    = [s["drift_combined_p"] for s in ordered]
    commits   = [s["commits_analyzed"] for s in ordered]
    windows   = [s["windows_analyzed"] for s in ordered]
    n_cps     = [s["n_level_a_change_points"] for s in ordered]

    def sig_color(p):
        if p is None: return "#bdc3c7"
        if p < 0.01: return "#c0392b"
        if p < 0.05: return "#e67e22"
        if p < 0.10: return "#f1c40f"
        return "#27ae60"

    p_labels  = [f"{p:.4f}" if p else "n/a" for p in p_vals]
    sig_flags = [
        "★★★" if p and p < 0.01 else
        "★"   if p and p < 0.05 else
        "~"   if p and p < 0.10 else
        "ns"  if p else "—"
        for p in p_vals
    ]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Fisher combined p-value (Level A only)",
                        "Level-A change points detected"),
        column_widths=[0.55, 0.45],
    )

    # Left: p-values
    p_plot = [p if p else 1.0 for p in p_vals]
    fig.add_trace(go.Bar(
        y=names, x=p_plot,
        orientation="h",
        marker_color=[sig_color(p) for p in p_vals],
        text=[f"{lbl} {flag}" for lbl, flag in zip(p_labels, sig_flags)],
        textposition="outside",
        hovertemplate="%{y}<br>Fisher p = %{text}<extra></extra>",
    ), row=1, col=1)
    fig.add_vline(x=0.05, line_dash="dash", line_color="#2c3e50",
                  line_width=1.5, row=1, col=1,
                  annotation_text="α=0.05", annotation_position="top")
    fig.update_xaxes(range=[0, 1.15], title_text="Fisher p",
                     tickvals=[0, 0.05, 0.25, 0.5, 1.0], row=1, col=1)

    # Right: change points
    fig.add_trace(go.Bar(
        y=names, x=n_cps,
        orientation="h",
        marker_color=[sig_color(p) for p in p_vals],
        text=[str(n) for n in n_cps],
        textposition="outside",
    ), row=1, col=2)
    fig.update_xaxes(title_text="# Level-A change points", row=1, col=2)

    fig.update_layout(
        title=(
            "Real Level-A process drift — 9 developers · 2018–2025 · "
            "Mann-Whitney U + Fisher's method<br>"
            "<sup>Red=p<0.01  Orange=p<0.05  Yellow=p<0.10  Green=ns  Grey=insufficient windows</sup>"
        ),
        showlegend=False,
        template="plotly_white",
        height=480,
    )
    fig.show()

    print(f"\n{'Developer':<30} {'Commits':>7} {'Wins':>5} {'Fisher p':>9} {'Sig':>4} {'CPs-A':>6}")
    print("─" * 62)
    for s, flag in zip(ordered, sig_flags):
        p = s['drift_combined_p']
        p_str = f"{p:.4f}" if p else "n/a"
        print(f"{s['display_name'][:29]:<30} {s['commits_analyzed']:>7} "
              f"{s['windows_analyzed']:>5} {p_str:>9} {flag:>4} {s['n_level_a_change_points']:>6}")

### 5. Signal-Level Detail — Top 3 Drifters

For each developer with Fisher p < 0.01, we show the 6 Level-A signals:
baseline mean vs. recent mean, direction and individual p-value.

This is the data that drives the combined test — showing which signals actually moved.

In [ ]:
LEVEL_A_SIGNALS = [
    "median_files_per_commit",
    "large_commit_ratio",
    "cross_module_ratio",
    "refactor_ratio",
    "median_inter_commit_hours",
    "commits_per_week",
]
SIG_LABELS = [
    "Files/commit",
    "Large commit ratio",
    "Cross-module ratio",
    "Refactor ratio",
    "Inter-commit hours",
    "Commits/week",
]

if summary_path.exists():
    strong_drifters = [s for s in summary if s.get("drift_combined_p") and s["drift_combined_p"] < 0.01]
    strong_drifters.sort(key=lambda s: s["drift_combined_p"])

    profiles = {}
    for dev in strong_drifters:
        login = dev["login"]
        p = REPORTS_DIR / f"{login}.json"
        if p.exists():
            profiles[dev["display_name"]] = json.loads(p.read_text())

    fig = make_subplots(
        rows=1, cols=len(profiles),
        subplot_titles=list(profiles.keys()),
    )

    for col_idx, (name, profile) in enumerate(profiles.items(), start=1):
        dr = profile.get("drift_result", {})
        if not dr:
            continue
        signals_data = {s["signal"]: s for s in dr.get("signals", []) if s.get("signal_level") == "A"}

        for row_sig, label in zip(LEVEL_A_SIGNALS, SIG_LABELS):
            sd = signals_data.get(row_sig)
            if not sd:
                continue
            baseline = sd["baseline_mean"]
            recent   = sd["recent_mean"]
            is_sig   = sd["significant_at_05"]

            bar_color_base   = "#4a90d9"
            bar_color_recent = "#e74c3c" if is_sig else "#95a5a6"

            fig.add_trace(go.Bar(
                name=f"{label} — baseline",
                x=[label], y=[baseline],
                marker_color=bar_color_base,
                opacity=0.8,
                showlegend=(col_idx == 1),
                legendgroup="baseline",
                legendgrouptitle_text="Baseline" if col_idx == 1 else None,
            ), row=1, col=col_idx)

            fig.add_trace(go.Bar(
                name=f"{label} — recent" + (" ★" if is_sig else ""),
                x=[label], y=[recent],
                marker_color=bar_color_recent,
                opacity=0.9,
                showlegend=False,
            ), row=1, col=col_idx)

    fig.update_layout(
        title="Level-A signals: baseline (blue) vs recent (red=significant, grey=not) — strong drifters only",
        barmode="overlay",
        template="plotly_white",
        height=450,
        showlegend=False,
    )
    fig.update_xaxes(tickangle=-35)
    fig.show()

### 6. BehaviorWindow Timeline — Real Data

Load a full profile from JSON and plot the Level-A signal trajectories
across all quarterly windows, with LLM milestones overlaid.

In [ ]:
# Choose which developers to plot
PLOT_DEVS = [
    ("gaearon",    "Dan Abramov",  "#e74c3c"),
    ("Rich-Harris","Rich Harris",  "#27ae60"),
    ("sindresorhus","Sindre Sorhus","#f39c12"),
]
SIGNAL_TO_PLOT = "commits_per_week"

fig = go.Figure()

for label, dt in LLM_MILESTONES.items():
    if label not in MILESTONE_COLORS:
        continue
    fig.add_vline(
        x=dt.timestamp() * 1000,
        line_dash="dot", line_color=MILESTONE_COLORS[label], line_width=1,
        annotation_text=label.split()[-1],
        annotation_position="top",
        annotation_font_size=9,
    )

for login, display, color in PLOT_DEVS:
    profile_path = REPORTS_DIR / f"{login}.json"
    if not profile_path.exists():
        print(f"⚠ {login}.json not found")
        continue

    profile = json.loads(profile_path.read_text())
    tl      = profile.get("behavior_timeline", [])
    dr      = profile.get("drift_result")
    p_val   = dr["combined_p_value"] if dr else None

    xs = [datetime.fromisoformat(w["period_start"]) for w in tl]
    ys = [w.get(SIGNAL_TO_PLOT, 0) for w in tl]

    legend_name = f"{display}" + (f" (p={p_val:.4f})" if p_val else " (n/a)")

    fig.add_trace(go.Scatter(
        x=xs, y=ys,
        mode="lines+markers",
        name=legend_name,
        line=dict(color=color, width=2),
        marker=dict(size=5),
    ))

    # Level-A change points from JSON
    cps_a = [cp for cp in profile.get("change_points", [])
             if cp.get("signal_level") == "A" and cp.get("signal") == SIGNAL_TO_PLOT]
    for cp in cps_a:
        cp_dt = datetime.fromisoformat(cp["date"])
        cp_y  = cp.get("value_after", 0)
        fig.add_trace(go.Scatter(
            x=[cp_dt], y=[cp_y + 0.2],
            mode="markers",
            marker=dict(symbol="triangle-down", size=14, color=color,
                        line=dict(width=1, color="white")),
            showlegend=False,
            hovertext=f"{display}: Δ {cp['magnitude']:.3f}  {cp.get('nearest_known_event','')}",
        ))

fig.update_layout(
    title=f"Real data: {SIGNAL_TO_PLOT} — quarterly windows 2018–2025<br>"
          "<sup>▼ = Level-A change point detected by PELT or CUSUM</sup>",
    xaxis_title="Quarter",
    yaxis_title=SIGNAL_TO_PLOT.replace("_", " "),
    legend=dict(orientation="h", y=-0.18),
    template="plotly_white",
    height=500,
)
fig.show()

### 7. Radar Chart — Style Profile (Level C)

A Level-C fingerprint radar chart: 6 style signals for an organic human profile
vs a synthetic LLM-assisted profile.

> **Reminder:** Level-C signals are calibrated on complete code files. On commit diffs,
> they have reduced sensitivity and should not be used as primary evidence. Rich Harris,
> who showed +6.8 style drift in v1, shows **no significant process drift** at Level A.

In [ ]:
radar_signals  = ["Comment<br>density", "Docstring<br>coverage", "Identifier<br>verbosity",
                   "Error<br>handling", "Type hint<br>usage", "Commit<br>structure"]
organic_values = [0.08, 0.15, 0.42, 0.04, 0.04, 0.30]
llm_values     = [0.22, 0.68, 0.66, 0.09, 0.20, 0.75]

def _radar(values, name, color, dash="solid"):
    v = values + [values[0]]
    s = radar_signals + [radar_signals[0]]
    return go.Scatterpolar(
        r=v, theta=s, name=name, fill="toself",
        line=dict(color=color, width=2, dash=dash),
        fillcolor=color, opacity=0.15,
    )

fig = go.Figure([
    _radar(organic_values, "Organic style (pre-2022 baseline)", "#4a90d9"),
    _radar(llm_values,     "LLM-assisted style (synthetic)",    "#e74c3c", dash="dot"),
])
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 0.85])),
    title="Level-C style fingerprint: organic vs LLM-assisted (synthetic calibration)",
    legend=dict(orientation="h", y=-0.12),
    template="plotly_white",
    height=460,
)
fig.show()

print("⚠ These Level-C radar values are calibrated on full code files, not commit diffs.")
print("  Real commit-diff scores fall in the 3–15/100 range due to signal dilution.")
print("  Do not use Level-C scores as primary evidence — see CRITIQUE.md.")

---
## Part C — Static Figures

The `generate_figures.py` script produces 4 publication-quality PNG figures
from the real profiles. Run it once after `run_analysis.py`:

```bash
python generate_figures.py
```

The cells below display the pre-generated figures inline.

In [ ]:
from IPython.display import display, Image
from pathlib import Path

FIGS = [
    ("../docs/img/level_a_heatmap.png",
     "Level-A Signal Heatmap — Δ% baseline→recent, significance per signal"),
    ("../docs/img/activity_timeline.png",
     "Commit Activity Timeline — commits/week, 2018–2025"),
    ("../docs/img/changepoint_calendar.png",
     "Change-Point Calendar — when did each developer's process shift?"),
    ("../docs/img/process_scatter.png",
     "Process Scatter — baseline vs. recent commits/week"),
]

for path, caption in FIGS:
    p = Path(path)
    if p.exists():
        print(f"\n{caption}")
        display(Image(filename=str(p), width=900))
    else:
        print(f"⚠ {p} not found — run: python generate_figures.py")

### 8. Reproduce — Fetch New Profiles

To re-fetch all 9 profiles from the GitHub API:

```bash
export GITHUB_TOKEN=ghp_...
python run_analysis.py --force          # all 9 developers
python run_analysis.py --logins gaearon --force  # single developer
```

To add a developer:

```yaml
# configs/developers.yaml
- github_login: your-target
  display_name: "Developer Name"
  primary_language: python
  repos:
    - owner/repo
```

The pipeline produces a `DevProfile` JSON at `reports/real/<login>.json` with:
- `behavior_timeline` — list of quarterly `BehaviorWindow` objects
- `change_points` — list of `ChangePoint` objects (Level A/B/C, method, nearest LLM event)
- `drift_result` — `DriftResult` with per-signal Mann-Whitney p-values and Fisher combined p

Full methodology: [METHODOLOGY.md](../METHODOLOGY.md)  
Per-developer findings: [FINDINGS.md](../FINDINGS.md)  
Research agenda: [RESEARCH_AGENDA.md](../RESEARCH_AGENDA.md)